In [ ]:
# ---------------- Imports ----------------
import os
import json

import pandas as pd
import yaml



In [ ]:
# ---------------- Args ----------------
FEVER_TRAIN_NAME = "train.jsonl"
FEVER_DEV_NAME = "shared_task_dev.jsonl"
FEVEROUS_TRAIN_NAME = "feverous_train_challenges.jsonl"
FEVEROUS_DEV_NAME = "feverous_dev_challenges.jsonl"



In [ ]:
# ---------------- Config ----------------

with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROJ_STORE = config["paths"]["proj-store"]
DATA_FOLDER = os.path.join(PROJ_STORE, "data")

EXTERNAL_SETS_FOLDER = os.path.join(DATA_FOLDER, "external")

# FEVER
FEVER_PATH = os.path.join(EXTERNAL_SETS_FOLDER, "fever")
FEVER_TRAIN_PATH = os.path.join(FEVER_PATH, FEVER_TRAIN_NAME)
FEVER_DEV_PATH = os.path.join(FEVER_PATH, FEVER_DEV_NAME)

# FEVEROUS
FEVEROUS_PATH = os.path.join(EXTERNAL_SETS_FOLDER, "feverous")
FEVEROUS_TRAIN_PATH = os.path.join(FEVEROUS_PATH, FEVEROUS_TRAIN_NAME)
FEVEROUS_DEV_PATH = os.path.join(FEVEROUS_PATH, FEVEROUS_DEV_NAME)

# OUTPUT
OUTPUT_DIR = os.path.join(DATA_FOLDER, "index")
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "combined-claims-full-index.json") # file extension added on output time.



In [ ]:
def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows

In [ ]:
# ---------------- Build Index ----------------

index = {}

ALLOWED_LABELS = {"SUPPORTS", "REFUTES"}

# Load both datasets
fever_train = load_jsonl(FEVER_TRAIN_PATH)
fever_dev = load_jsonl(FEVER_DEV_PATH)

feverous_train = load_jsonl(FEVEROUS_TRAIN_PATH)
feverous_dev = load_jsonl(FEVEROUS_DEV_PATH)


all_fever = fever_train + fever_dev
all_feverous = feverous_train + feverous_dev


# ---------- FEVER ----------

for ex in all_fever:

    if ex["label"] not in ALLOWED_LABELS:
        continue

    raw_id = str(ex["id"]).strip()
    if not raw_id:
        continue

    claim_id = f"fever-{raw_id}"

    pages = set()

    for evidence_set in ex["evidence"]:
        for ev in evidence_set:

            page = ev[2]

            if page:
                pages.add(page)

    index[claim_id] = sorted(pages)


# ---------- FEVEROUS ----------

for ex in all_feverous:

    if ex["label"] not in ALLOWED_LABELS:
        continue

    raw_id = str(ex["id"]).strip()
    if not raw_id:
        continue

    claim_id = f"feverous-{raw_id}"

    pages = set()

    for ev_block in ex["evidence"]:

        context = ev_block.get("context", {})

        for refs in context.values():
            for ref in refs:

                if ref.endswith("_title"):

                    page = ref[:-6]              # remove _title
                    page = page.replace(" ", "_")

                    pages.add(page)

    index[claim_id] = sorted(pages)


print(f"Indexed examples: {len(index)}")



In [ ]:
# ---------------- Save ----------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(index, f, indent=2)

print(f"Saved index → {OUTPUT_FILE}")

